# DPPUv2 Paper 03 — Interactive Visualizer

**Purpose:** Interactive exploration of the effective potential landscape and spin-2 spectrum in the $S^3 \times S^1$ geometry with squash deformation $\varepsilon$ and mixed torsion (axial $\eta$ + vector $V$).

## What you can explore

| Panel | Description |
|---|---|
| **Phase Diagram** | Stability classification in the $(\varepsilon, \eta)$ plane.  Extends the $(V, \eta)$ diagram of paper02 to the squash dimension. |
| **V_eff(r)** | Effective potential at the selected $(\varepsilon, \eta)$ point. |
| **Spin-2 spectrum** | Three spin-2 Hessian branches vs. $\varepsilon$ at the selected $\eta$: $m=0$ (squash, $T_1$), $m=\pm 2$ (shear, $T_2$), $m=\pm 1$ (mixing, $T_4$).  Zero crossing of $m=0$ marks $\varepsilon_c$. |

## Fixed-parameter sliders

| Slider | Symbol | Range | Default |
|---|---|---|---|
| Nieh–Yan coupling | $\theta_{\rm NY}$ | $[0, 10]$ | $1.0$ |
| Vector torsion | $V$ | $[0, 15]$ | $0.0$ |
| Weyl coupling | $\alpha$ | $[-5, 3]$ | $0.0$ |

Changes to these sliders take effect on the next **"Draw Phase Diagram"** click.

## Engine

- Topology: $S^3 \times S^1$  
- Active DOF: squash $\varepsilon$ (axisymmetric)  
- Torsion mode: MX ($\eta \neq 0$, $V \neq 0$ simultaneously)  
- NY variant: FULL  
- Fixed: $L = 1$, $\kappa = 1$  

**Requirements:** Python 3.10+, `ipywidgets`, `ipympl`, `numpy`, `scipy`, `sympy`, `dppu` package

## Note on shear and twist parameters

This visualizer deliberately **excludes** the shear $s$ (spin-2, $T_2$ / $m = \pm 2$ direction) and twist $\omega$ (spin-1 fiber deformation) parameters.  The reasons are:

### Shear $s$

From the cross-Hessian analysis (paper03 Appendix H.2), the spin-2 sector is **decoupled from the torsion sector at quadratic order** around the isotropic point: all 10 spin-2 $\times$ torsion cross terms vanish.  Furthermore, $s = 0$ is the natural axisymmetric slice: the first condensation is the $T_1$ (squash singlet, $m = 0$) mode, which is precisely the $\varepsilon$ direction explored here.  Including $s$ would require the `FullyAnisotropicS3S1Engine` (substantially heavier) and would add a second deformation axis that does not interact with $\eta$ at this order.

### Twist $\omega$

The paper03 main result (Sec. 3) shows that $\omega = 0$ is always the global minimum of the twist potential: the chiral equilibrium is geometrically enforced.  Concretely,
$$m^2_\omega(r) = \frac{48\pi^2 L^3}{\kappa^2 r} > 0 \quad \forall\, r > 0.$$
Adding an $\omega$ slider would show the potential rising away from $\omega = 0$, but the **phase boundaries in the $(\varepsilon, \eta)$ plane are not affected**: twist does not shift $\varepsilon_c$.

In [ ]:
%matplotlib widget

import sys
import os
from pathlib import Path

# ── Path setup ────────────────────────────────────────────────────────────────
# Locate the bundled script root whether Jupyter starts in the paper root
# or in script/scripts/visualize/.
def _find_script_root():
    cwd = Path.cwd().resolve()
    bases = [cwd, *cwd.parents]
    candidates = bases + [base / 'script' for base in bases]
    for candidate in candidates:
        if (candidate / 'dppu').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise RuntimeError('Could not locate paper03 script root containing dppu/ and scripts/.')

_script_root_path = _find_script_root()
_script_root = str(_script_root_path)
_viewer_dir  = str(_script_root_path / 'scripts' / 'visualize')

for _p in [_script_root, _viewer_dir]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f'dppu root : {_script_root}')
print(f'viewer dir: {_viewer_dir}')

## Engine initialisation

The cell below runs the `S3×S¹` squash engine with Mode.MX and lambdifies `V_eff`.  
This is done **once** — subsequent slider changes do not re-run the engine.  
Expected time: **10 – 60 seconds** depending on hardware.

In [ ]:
from DPPUv2_interactive_viewer_p03 import DPPUv2Paper03Viewer

vfunc, engine_info = DPPUv2Paper03Viewer.setup_engine(_script_root)

## Launch viewer

**Controls:**
- Adjust **Fixed Parameters** (θ_NY, V, α) and click **"Draw Phase Diagram"** to compute the $(\varepsilon, \eta)$ stability map.
- **Click** anywhere on the phase diagram, or use the **epsilon / eta sliders**, to place the point marker.
- Click **"Update Potential"** to plot $V_{\rm eff}(r)$ at the selected point.
- Click **"Update Spectrum"** to compute the three spin-2 branches vs. $\varepsilon$ at the selected $\eta$ (~2–4 s).

**Phase colour key** (same as paper02):

| Colour | Type | Condition | Description |
|---|---|---|---|
| Light green | Type I | $\alpha \approx 0$ | Stable well with barrier ($V_{\rm eff}(r_0) < 0$) |
| Light yellow | Type II | $\alpha \approx 0$ | Rolling — local min exists but no barrier near $r=0$ |
| Light gray | Type III | $\alpha \approx 0$ | No local minimum |
| Light red | Type G | $\alpha > 0$ | Metastable minimum (ghost regime) |
| Light gray (hatched) | Type III-G | $\alpha > 0$ | No local minimum, unbounded below at $r \to 0$ |
| Light blue | Type I-W | $\alpha < 0$ | Double well — two or more local minima (repulsive wall) |
| Light orange | Type II-W | $\alpha < 0$ | Single well — one local minimum (repulsive wall) |
| Navy dashed | — | — | $H_{\varepsilon\varepsilon} = 0$ (spin-2 $\varepsilon_{c+}$ boundary) |
| Brown dashed | — | — | $H_{ss} = 0$ (spin-2 $\varepsilon_{c-}$ boundary) |

In [ ]:
viewer = DPPUv2Paper03Viewer(vfunc, engine_info)
viewer.display()